In [29]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [30]:
ticker = "BBDC4.SA"
dados = yf.download(ticker, period="6mo")
dados.columns = dados.columns.droplevel('Ticker')
df = dados
df = df.dropna(subset=['Close', 'High', 'Low', 'Open'])

[*********************100%***********************]  1 of 1 completed


In [31]:
# ==========================================
# BLOCO 1: CALCULO DO MACD
# ==========================================
# EMA12 e EMA26 (ajustado=False simula fielmente a fórmula clássica de Excel)
df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA26'] = df['Close'].ewm(span=26, adjust=False).mean()

# MACD (Linha rápida)
df['MACD'] = df['EMA12'] - df['EMA26']

# Linha de Sinal (Média de 9 períodos do próprio MACD)
df['Signal(9)'] = df['MACD'].ewm(span=9, adjust=False).mean()

# Histograma (Diferença entre o MACD e o Sinal)
df['Histograma'] = df['MACD'] - df['Signal(9)']

# ==========================================
# BLOCO 2: CALCULO DO IFR (RSI 14)
# ==========================================
# Variação diária do preço de fechamento
df['Variacao'] = df['Close'].diff()

# Identificar Ganhos e Perdas individuais
df['Ganho'] = df['Variacao'].apply(lambda x: x if x > 0 else 0.0)
df['Perda'] = df['Variacao'].apply(lambda x: abs(x) if x < 0 else 0.0)

# Média de Ganhos e Média de Perdas utilizando a média móvel de Wilder (padrão do IFR)
# O primeiro valor é uma média simples, e os seguintes usam suavização exponencial adaptada
df['Media_Ganhos'] = df['Ganho'].ewm(alpha=1/14, adjust=False, min_periods=14).mean()
df['Media_Perdas'] = df['Perda'].ewm(alpha=1/14, adjust=False, min_periods=14).mean()

# Razão de Força Relativa (RS)
df['RS'] = df['Media_Ganhos'] / df['Media_Perdas']

# Índice de Força Relativa (IFR)
df['IFR'] = 100 - (100 / (1 + df['RS']))

# Calculando média EMA(9) de IFR
df['IFR(9)'] = df['IFR'].ewm(span=9, adjust=False).mean()

# Linhas de Suporte e Resistência do IFR
df['LS'] = 70
df['LI'] = 30

# ==========================================
# BLOCO 3: CÁLCULO DAS BANDAS DE BOLLINGER (20 períodos, 2 desvios)
# ==========================================
periodo_bb = 20
desvios_bb = 2

# 1. Banda Central: Média Móvel Simples de 20 dias
df['BB_Central'] = df['Close'].rolling(window=periodo_bb, min_periods=periodo_bb).mean()

# 2. Desvio Padrão de 20 dias
df['BB_Desvio'] = df['Close'].rolling(window=periodo_bb, min_periods=periodo_bb).std(ddof=0)

# 3. Banda Superior: Média + (2 * Desvio Padrão)
df['BB_Superior'] = df['BB_Central'] + (desvios_bb * df['BB_Desvio'])

# 4. Banda Inferior: Média - (2 * Desvio Padrão)
df['BB_Inferior'] = df['BB_Central'] - (desvios_bb * df['BB_Desvio'])

# Remover a coluna do desvio
df = df.drop(columns=['BB_Desvio'])

# ==========================================
# BLOCO 4: RETRAÇAMENTO DE FIBONACCI (Últimos 6 meses)
# ==========================================
# 1. Encontrar o maior topo (Máxima) e o menor fundo (Mínima) do período
topo = df['High'].max() if 'High' in df.columns else dados['High'].max()
fundo = df['Low'].min() if 'Low' in df.columns else dados['Low'].min()

# 2. Calcular a amplitude do movimento (distância entre o topo e o fundo)
amplitude = topo - fundo

# 3. Criar as colunas estáticas de Fibonacci
df['Fibo_100'] = topo
df['Fibo_618'] = topo - (0.382 * amplitude) # Retração de 61.8%
df['Fibo_500'] = topo - (0.500 * amplitude) # Retração de 50.0%
df['Fibo_382'] = topo - (0.618 * amplitude) # Retração de 38.2%
df['Fibo_236'] = topo - (0.764 * amplitude) # Retração de 23.6%
df['Fibo_0']   = fundo

# ==========================================
# BLOCO 5: CÁLCULO DO ATR (Average True Range - 14 Períodos)
# ==========================================
periodo_atr = 14

# 1. Obter o fechamento do dia anterior
df['Close_Anterior'] = df['Close'].shift(1)

# 2. Calcular as 3 variações possíveis que compõem o True Range
df['TR1'] = df['High'] - df['Low']                      # Máxima atual - Mínima atual
df['TR2'] = (df['High'] - df['Close_Anterior']).abs()   # Valor absoluto de: Máxima atual - Fechamento anterior
df['TR3'] = (df['Low'] - df['Close_Anterior']).abs()    # Valor absoluto de: Mínima atual - Fechamento anterior

# 3. O True Range (TR) é o maior valor entre as 3 variações
df['TR'] = df[['TR1', 'TR2', 'TR3']].max(axis=1)

# 4. O ATR é a Média Móvel de Wilder do TR (peso 1/14)
df['ATR'] = df['TR'].ewm(alpha=1/periodo_atr, adjust=False, min_periods=periodo_atr).mean()

# Limpar as colunas intermediárias
df = df.drop(columns=['Close_Anterior', 'TR1', 'TR2', 'TR3', 'TR'])

In [32]:
# Exportar o DataFrame
df.to_excel(f"Analise_Tecnica_{ticker}.xlsx", sheet_name="Indicadores")

In [ ]:
#----------------------------------------------------------------------------------
# Candlestick, Retração de Fibonacci e Curvas de Bollinger
#----------------------------------------------------------------------------------
# 1. Criar a estrutura de subplots (2 linhas e 1 coluna)
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, # shared_xaxes=True garante que ao dar zoom no preço, o volume acompanha o mesmo período automaticamente
    vertical_spacing=0.05, # Espaço pequeno entre os dois painéis
    row_heights=[0.75, 0.25] # 75% da tela para o Preço, 25% para o Volume
)

# Exportar gráficos para html
# fig.write_html("grafico.html", auto_open=True)
# pio.renderers.default = "browser"

# PAINEL 1 (LINHA 1): CANDLESTICK + BOLLINGER
# Banda Superior
fig.add_trace(go.Scatter(
    x=df.index, y=df['BB_Superior'], name='Bollinger Sup',
    line=dict(color='rgba(173, 181, 189, 0.6)', width=1.5, dash='dash'),
    hoverinfo='skip'
), row=1, col=1)

# Banda Inferior com Preenchimento
fig.add_trace(go.Scatter(
    x=df.index, y=df['BB_Inferior'], name='Bollinger Inf',
    line=dict(color='rgba(173, 181, 189, 0.6)', width=1.5, dash='dash'),
    fill='tonexty', fillcolor='rgba(173, 181, 189, 0.08)',
    hoverinfo='skip'
), row=1, col=1)

# Banda Central
fig.add_trace(go.Scatter(
    x=df.index, y=df['BB_Central'], name='Média Central (20)',
    line=dict(color='#2196f3', width=1.5)
), row=1, col=1)

# Candlestick
fig.add_trace(go.Candlestick(
    x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'], name='Preço',
    increasing_line_color='#26a69a', decreasing_line_color='#ef5350',
    increasing_fillcolor='#26a69a', decreasing_fillcolor='#ef5350'
), row=1, col=1)

# PAINEL 2 (LINHA 2): VOLUME
# Definindo cores dinâmicas para o volume (Verde se o dia fechou em alta, Vermelho em baixa)
cores_volume = ['#26a69a' if c >= o else '#ef5350' for o, c in zip(df['Open'], df['Close'])]

fig.add_trace(go.Bar(
    x=df.index, 
    y=df['Volume'], 
    name='Volume',
    marker_color=cores_volume,
    opacity=0.8
), row=2, col=1)

# Adicionando os níveis de Fibonacci no gráfico
niveis_fibo = {
    'Fibo_100': {'nome': 'Fibo 100% (Topo)', 'cor': '#ef5350'}, # Vermelho
    'Fibo_618': {'nome': 'Fibo 61.8%',       'cor': '#ff9800'}, # Laranja
    'Fibo_500': {'nome': 'Fibo 50.0%',       'cor': '#4caf50'}, # Verde
    'Fibo_382': {'nome': 'Fibo 38.2%',       'cor': '#2196f3'}, # Azul
    'Fibo_236': {'nome': 'Fibo 23.6%',       'cor': '#9c27b0'}, # Roxo
    'Fibo_0':   {'nome': 'Fibo 0% (Fundo)',  'cor': '#757575'}  # Cinza
}

for coluna, config in niveis_fibo.items():
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df[coluna],
        name=config['nome'],
        line=dict(color=config['cor'], width=1.2, dash='dot'),
        mode='lines'
    ))

# Customização do Layout
fig.update_layout(
    title=f"Candlestick, Retração de Fibonacci e Bandas de Bollinger - {ticker}",
    yaxis_title="Preço (R$)", # Título do eixo Y do primeiro painel
    yaxis2_title="Volume", # Título do eixo Y do segundo painel
    template="plotly_white",
    xaxis_rangeslider_visible=False, # Remove o slider para otimizar espaço
    hovermode="x unified", # Passar o mouse mostra o preço e o volume do dia ao mesmo tempo
    height=650, # Define uma altura para visualizar os dois gráficos
    showlegend=False
)

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]), # Esconde finais de semana
        dict(values=["2026-01-01","2026-02-16","2026-02-17","2026-04-03","2026-04-21","2026-05-01"]) # Esconde feriados
    ]
)

fig.show()

#----------------------------------------------------------------------------------
# MACD 
#----------------------------------------------------------------------------------
# 1. Criar a figura
fig = make_subplots(rows=1, cols=1)

# 2. Adicionar o Histograma (em colunas/barras)
# Definir a cor com base em valores positivos (verde) ou negativos (vermelho)
fig.add_trace(go.Bar(
        x=df.index,
        y=df['Histograma'],
        name='Histograma',
        marker_color=['#26a69a' if val >= 0 else '#ef5350' for val in df['Histograma']],
        opacity=0.6
    )
)

# 3. Adicionar a linha do MACD (Linha Rápida)
fig.add_trace(go.Scatter(
        x=df.index,
        y=df['MACD'],
        name='MACD',
        line=dict(color='#2196f3', width=2) # Azul
    )
)

# 4. Adicionar a linha do Sinal (Linha Lenta)
fig.add_trace(go.Scatter(
        x=df.index,
        y=df['Signal(9)'],
        name='Sinal (9)',
        line=dict(color='#ff9800', width=1.5, dash='dot') # Laranja pontilhado
    )
)

# 5. Customizar o layout do gráfico
fig.update_layout(
    title=f"Indicador MACD - {ticker}",
    xaxis_title="Data",
    yaxis_title="Valor",
    barmode='relative', # Garante que as barras partam do zero para cima ou para baixo
    xaxis_rangeslider_visible=False, # Desativa o slider para o gráfico não ficar espremido
    template="plotly_white", # Fundo branco para melhor contraste
    hovermode="x unified", # Mostra todos os valores ao passar o mouse em uma data
    showlegend=False
)

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]), # Esconde finais de semana
        dict(values=["2026-01-01","2026-02-16","2026-02-17","2026-04-03","2026-04-21","2026-05-01"]) # Esconde feriados específicos (Ex: Ano Novo e Sexta-feira Santa)
    ]
)

# Exibir o gráfico
fig.show()

#----------------------------------------------------------------------------------
# IFR
#----------------------------------------------------------------------------------
# 1. Criar a figura para o IFR
fig = go.Figure()

# 2. Adicionar as linhas de Limite (LS e LI)
fig.add_trace(go.Scatter(
        x=df.index,
        y=df['LS'],
        name='Limite Superior (70)',
        line=dict(color='rgba(239, 83, 80, 0.5)', width=1.5, dash='dash'), # Vermelho pontilhado suave
        showlegend=True
    )
)

fig.add_trace(go.Scatter(
        x=df.index,
        y=df['LI'],
        name='Limite Inferior (30)',
        line=dict(color='rgba(38, 166, 154, 0.5)', width=1.5, dash='dash'), # Verde pontilhado suave
        showlegend=True
    )
)

# 3. Adicionar a linha principal do IFR
fig.add_trace(go.Scatter(
        x=df.index,
        y=df['IFR'],
        name='IFR',
        line=dict(color="#b91515", width=2), # Roxo, cor clássica para o IFR
        mode='lines'
    )
)

# 3. Adicionar a linha principal do IFR(9 períodos)
fig.add_trace(go.Scatter(
        x=df.index,
        y=df['IFR(9)'],
        name='IFR (9)',
        line=dict(color='#9c27b0', width=2), # Roxo, cor clássica para o IFR
        mode='lines'
    )
)

# 4. Customizar o layout e fixar o eixo Y entre 0 e 100
fig.update_layout(
    title=f"Índice de Força Relativa (IFR / IFR(9)) - {ticker}",
    xaxis_title="Data",
    yaxis_title="Valor do IFR",
    yaxis=dict(
        range=[0, 100], # Oscilador fechado entre 0 e 100
        tickvals=[0, 30, 50, 70, 100] # Destaca os níveis no eixo Y
    ),
    template="plotly_white",
    hovermode="x unified",
    xaxis_rangeslider_visible=False,
    showlegend=False
)

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]), # Esconde finais de semana
        dict(values=["2026-01-01","2026-02-16","2026-02-17","2026-04-03","2026-04-21","2026-05-01"]) # Esconde feriados
    ]
)

fig.show()

#----------------------------------------------------------------------------------
# Candlestick
#----------------------------------------------------------------------------------
# 1. Criar o gráfico de Candlestick original (Gráfico simples)
fig = go.Figure(data=[go.Candlestick(
    x=dados.index,
    open=dados['Open'],
    high=dados['High'],
    low=dados['Low'],
    close=dados['Close'],
    name=ticker
)])

# Personalizar o layout
fig.update_layout(
    title=f"Histórico de Candlestick - {ticker} (Últimos 6 meses)",
    xaxis_title="Data",
    yaxis_title="Preço (R$)",
    xaxis_rangeslider_visible=False,
    showlegend=False
)

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]), # Esconde finais de semana
        dict(values=["2026-01-01","2026-02-16","2026-02-17","2026-04-03","2026-04-21","2026-05-01"]) # Esconde feriados
    ]
)

fig.show()

#----------------------------------------------------------------------------------
# ATR
#----------------------------------------------------------------------------------
# 1. Inicializar a figura para o ATR
fig = go.Figure()

# 2. Adicionar a linha do ATR
fig.add_trace(go.Scatter(
        x=df.index,
        y=df['ATR'],
        name='ATR (14)',
        line=dict(color='#ff5722', width=2), # Cor laranja/coral bem visível
        mode='lines'
    )
)

# 3. Customizar o layout do gráfico
fig.update_layout(
    title=f"Indicador ATR (Average True Range - 14) - {ticker}",
    xaxis_title="Data",
    yaxis_title="Volatilidade Média (R$)",
    template="plotly_white",
    hovermode="x unified",
    xaxis_rangeslider_visible=False
)

# 4. Remover finais de semana e feriados
fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]), # Esconde finais de semana
        dict(values=["2026-01-01","2026-02-16","2026-02-17","2026-04-03","2026-04-21","2026-05-01"]) # Esconde feriados específicos
    ]
)

fig.show()